# Cross-Border Due Diligence with Claude + OpenRegistry MCP

A compliance analyst onboarding a new counterparty has to verify *who actually controls the entity* — not just the operating subsidiary that signs the contract, but the corporate parent two or three layers up, possibly in another country. Pulling that information manually means logging into UK Companies House for the operating subsidiary, then France RNE for the parent, then maybe Italy InfoCamere or Spain BORME for the next layer, each with its own field names and quirks. A junior analyst routinely spends 4–8 hours per entity, and they often miss a filing because they don't have credentials for one of the registries in the chain.

OpenRegistry is a free remote MCP server that exposes 27 national company registries through one Streamable HTTP endpoint. Connecting Claude to that endpoint via the Messages API's MCP connector lets the model run live, real-time queries against each registry — UK Companies House, France RNE, Germany Handelsregister, Italy InfoCamere via EU BRIS, and 23 others — and chain them together inside a single conversation.

**By the end of this cookbook, you'll be able to:**
- Connect Claude to a remote MCP server with no separate MCP client setup
- Search a UK-registered company by name and read its statutory profile from Companies House
- Pull the company's officers and Persons with Significant Control (PSCs) from the live filing
- Have Claude orchestrate a multi-step ownership-chain walk across jurisdictions and cite each fact back to its government filing

The same pattern extends to any other public-data MCP server — once you've done one, every additional registry is just another `mcp_servers` entry.

## Prerequisites

Before following this guide, ensure you have:

**Required Knowledge:**
- Python fundamentals — comfortable with functions and dictionaries
- Basic understanding of HTTP APIs and JSON

**Required Tools:**
- Python 3.11 or higher
- Anthropic API key ([get one here](https://console.anthropic.com))

**Optional:**
- Familiarity with the [Model Context Protocol](https://modelcontextprotocol.io/)
- An OpenRegistry account if you want to test the higher rate-limit tiers — but **everything in this cookbook works on the free anonymous tier**, no signup or token required

## Setup

Install the required dependencies:

In [ ]:
%%capture
%pip install -U anthropic python-dotenv

Add your Anthropic key to a local `.env` file (one line: `ANTHROPIC_API_KEY=sk-ant-...`), then load it and configure the client.

OpenRegistry's hosted endpoint is `https://openregistry.sophymarine.com/mcp`. We'll keep that as a constant alongside the model name so both are easy to swap.

In [ ]:
import anthropic
from dotenv import load_dotenv

load_dotenv()

MODEL = "claude-opus-4-7"
OPENREGISTRY_URL = "https://openregistry.sophymarine.com/mcp"
MCP_BETA = "mcp-client-2025-11-20"

client = anthropic.Anthropic()

## Step 1 — Connect Claude to OpenRegistry and see what tools come back

The Messages API takes two pieces of MCP configuration:

1. `mcp_servers` — an array of server URLs and (optional) auth tokens
2. `tools` — one `mcp_toolset` entry per server, optionally restricting which tools are exposed

For our first call we'll enable every tool OpenRegistry advertises and ask Claude to introspect — this is exactly what an end user would do the first time they connect.

We also set the `mcp-client-2025-11-20` beta header. The MCP connector is still in beta, and the header opts the request into the current toolset configuration shape.

In [ ]:
def ask(prompt: str, *, max_tokens: int = 4096) -> anthropic.types.Message:
    """Single-turn helper that sends a user message with OpenRegistry attached."""
    return client.beta.messages.create(
        model=MODEL,
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
        mcp_servers=[
            {
                "type": "url",
                "url": OPENREGISTRY_URL,
                "name": "openregistry",
            }
        ],
        tools=[{"type": "mcp_toolset", "mcp_server_name": "openregistry"}],
        betas=[MCP_BETA],
    )


def render(message: anthropic.types.Message) -> None:
    """Print Claude's text plus a one-line summary of every MCP tool call."""
    for block in message.content:
        kind = block.type
        if kind == "text":
            print(block.text)
        elif kind == "mcp_tool_use":
            print(f"  → tool: {block.name}({block.input})")
        elif kind == "mcp_tool_result":
            preview = str(block.content)[:240].replace("\n", " ")
            print(f"  ← result: {preview}")
    print(f"\n[stop_reason={message.stop_reason}]")


render(
    ask(
        "Which jurisdictions does OpenRegistry cover today, and which kinds of"
        " tools are exposed? Use list_jurisdictions and summarise — don't pull"
        " any company records yet."
    )
)

Claude makes a single `list_jurisdictions` call, gets back the capability matrix (which tools each registry supports, which file types they return), and summarises it in prose. Note that we never had to write a single line of MCP-protocol code — the `beta.messages.create` call did the JSON-RPC dance internally.

## Step 2 — Look up a UK-registered company

We'll use **Tesco PLC** (a large, stable LSE-listed retailer) as our running example. Its UK number is `00445790`, but in real life you'd start from a name. Asking Claude to disambiguate is the realistic flow:

In [ ]:
render(
    ask(
        "Find Tesco PLC on Companies House (jurisdiction gb). Confirm the"
        " company number and registered office address straight from the"
        " registry, and quote the exact upstream `company_status` value."
    )
)

Two things worth noticing:

- The *upstream* `company_status` field is preserved verbatim. OpenRegistry does not normalise it to a generic enum — the cookbook user keeps the registry's own taxonomy, which is what auditors usually want.
- The company number is the natural foreign key for everything else. From here, the same `00445790` flows into the officers, PSC, charges, and filings tools.

## Step 3 — Pull officers and Persons with Significant Control

Companies House publishes two distinct people-lists:

- **Officers**: the legal record of who is or has been a director or secretary
- **Persons with Significant Control (PSCs)**: the statutory beneficial-owner declaration (UK PSC Register, in force since 2016)

Both matter for due diligence — the officers tell you who's accountable on paper, the PSCs tell you who actually benefits from the company's profits. We'll let Claude pull both in one call:

In [ ]:
render(
    ask(
        "For Tesco PLC (gb/00445790), pull the current directors via"
        " get_officers and the persons with significant control via"
        " get_persons_with_significant_control. List each PSC with their"
        " `nature_of_control` exactly as the registry returns it."
    )
)

`nature_of_control` is the exact string the PSC Register stores — entries like `ownership-of-shares-75-to-100-percent` or `voting-rights-25-to-50-percent`. These strings are stable enough to dedupe across filings, which makes them useful for downstream pipelines.

## Step 4 — Walk the ownership chain across borders

This is where the agentic part earns its keep. A single open-ended prompt asks Claude to:

1. Identify any **corporate** PSCs (i.e. another company, not an individual)
2. Look up that company in *its* jurisdiction
3. Repeat until the ultimate beneficial owner is an individual or a regulated entity
4. Cite every step

We keep the prompt deliberately under-specified. With OpenRegistry's `list_jurisdictions` capability matrix already in context from earlier turns, Claude has enough information to plan the walk on its own. (For very deep chains you may want a second model turn or a higher `max_tokens`.)

In [ ]:
render(
    ask(
        "Pick a UK Ltd that is a non-trading subsidiary of an overseas parent —"
        " e.g. search Companies House for 'Apple Retail UK Limited' or"
        " 'Microsoft Limited'. For one such company:\n"
        "  1. Confirm the UK record (number, status, registered office).\n"
        "  2. Identify the corporate PSC.\n"
        "  3. Look up that parent in its own jurisdiction (use"
        " list_jurisdictions to confirm the relevant country code first).\n"
        "  4. Repeat one more layer if the parent itself has a corporate"
        " parent that is in one of OpenRegistry's covered jurisdictions.\n"
        "For every fact, cite the registry and identifier you got it from.",
        max_tokens=8192,
    )
)

Two production-honest observations:

- **Some BO registers are gated.** After CJEU ruling C-37/20 (November 2022) the German, Spanish, Italian, Dutch, Luxembourg, Austrian, Maltese and Portuguese beneficial-ownership registers became access-restricted. OpenRegistry doesn't proxy those — it returns a `501 alternative_url` pointing at the statutory portal AML-obliged entities use. Claude will surface that signal verbatim, which is what an auditor needs.
- **The walk can fan out wide.** In the worst case (e.g. an investment vehicle structured through Jersey, Cayman and Delaware) Claude will hit OpenRegistry's per-jurisdiction rate limits. The free anonymous tier caps cross-border fan-out to 3 countries / 60s — fine for a single ad-hoc query, but for batch onboarding you'll want a Pro account.

## Step 5 — Production considerations

**Authentication.** Anonymous calls use the IP-based rate limit (20/min). For per-user limits and higher cross-border fan-out, OpenRegistry uses OAuth 2.1 with Dynamic Client Registration (RFC 7591) — there is no API key to copy/paste. Once you've completed the OAuth flow (the [MCP inspector](https://github.com/modelcontextprotocol/inspector) walks you through it), pass the access token through to Claude:

```python
mcp_servers = [
    {
        "type": "url",
        "url": OPENREGISTRY_URL,
        "name": "openregistry",
        "authorization_token": OPENREGISTRY_OAUTH_TOKEN,
    }
]
```

**Tool budgeting.** OpenRegistry exposes ~30 tools. For latency-sensitive flows, allowlist just the ones you need with a `mcp_toolset` config:

```python
tools = [
    {
        "type": "mcp_toolset",
        "mcp_server_name": "openregistry",
        "default_config": {"enabled": False},
        "configs": {
            "search_companies": {"enabled": True},
            "get_company_profile": {"enabled": True},
            "get_persons_with_significant_control": {"enabled": True},
        },
    }
]
```

**Auditability.** Every OpenRegistry response keeps the registry's native `company_id` and (on the Enterprise tier) pre-synthesises a `source_url` field. That makes it trivial to drop the citation directly into a compliance report — the URL goes to the government's own filing portal, not an aggregator's mirror.

## Recap of what we did in this guide

We connected Claude Opus 4.7 to a hosted MCP server, walked one UK company end-to-end (profile → officers → PSCs), and let the model agentically chain queries across a corporate ownership structure. Specifically:

- Wired up the Messages API's `mcp_servers` parameter with no separate MCP client code
- Pulled live records from UK Companies House and saw the registry's own field names preserved verbatim
- Watched Claude plan a multi-step cross-border walk from a single open-ended prompt
- Saw how OpenRegistry signals statutorily-gated beneficial-ownership registers via a `501 alternative_url` so the agent knows where the AML wall is

**Where to go next:**

1. **Different jurisdiction shapes.** The non-UK registries each have their own quirks — German Handelsregister returns iXBRL filings, Spain BORME publishes daily change batches, Italy InfoCamere is fronted by EU BRIS and has different officer-list semantics. Read the [OpenRegistry capability matrix](https://openregistry.sophymarine.com/jurisdictions) for the full per-country table.
2. **Combine with other MCP servers.** Add a second entry to `mcp_servers` (e.g. a sanctions-list MCP) and Claude can cross-reference each PSC against the sanctions list in the same conversation.
3. **Production patterns.** For batch onboarding, see OpenRegistry's [skillpack](https://github.com/sophymarine/openregistry/tree/main/skills) — ten ready-to-drop Claude Agent Skills covering KYC, UBO walks, PEP screening, and corporate filing monitoring.